# 01 · Preparación y auditoría de datos — Reto AgroCebada 2026

**Objetivo.** Preparar el entorno de Colab y Drive, encontrar y verificar cada archivo de entrada, auditar cada conjunto de datos (rendimientos, polígonos de las parcelas, tablas satelitales PRO y Básico, rásteres de clima y topografía) y medir la referencia basada solo en la ubicación que todo modelo posterior debe superar.

**Este cuaderno no construye variables ni filtra datos.** Solo asigna tipos, valida y documenta. La limpieza empieza en el cuaderno 02.

**Antes de ejecutar**
1. En Google Drive, crea `MyDrive/AgroCebada/raw/` y copia ahí **todos** los archivos del reto.
2. Entorno de ejecución: basta con CPU. Ejecuta todas las celdas en orden (~3–6 min).

**Salidas**
| Archivo | Contenido |
|---|---|
| `interim/labels.parquet` | ID, área, rendimiento, CONJUNTO, `is_train` |
| `interim/plots.parquet` | GeoParquet: polígonos (EPSG:4326, 2D), municipio, estado, centroide métrico, área e identificador de celda climática |
| `interim/plots_attributes.csv` | Lo mismo sin geometría |
| `interim/pro_obs.parquet`, `interim/basic_obs.parquet` | Tablas satelitales con fechas bien interpretadas y nombres de sensor normalizados. **Se conservan todas las filas.** |
| `qa/file_inventory.csv`, `qa/raster_inventory.csv`, `qa/sat_clear_obs_per_month_*.csv` | Inventarios |
| `qa/centroid_nodata_fixes.csv` | Parcelas cuyo centroide cae en un píxel sin dato, por capa, y cómo se resolvió cada caso |
| `models/baselines_location.json` | Referencias basadas solo en la ubicación (dejando una fuera) |
| `qa/audit_report.md` | Todo lo anterior en un solo reporte |
| `qa/entorno_pip_freeze.txt` | Versiones exactas de los paquetes del entorno de ejecución |

In [1]:
# 1 · Paquetes que Colab no trae instalados (geopandas, shapely, pandas y pyarrow ya vienen)
!pip install -q rasterio pyogrio

In [2]:
# 2 · Librerías, Drive y configuración
import os, re, sys, json, hashlib, unicodedata, warnings, subprocess, platform
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 50)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------- CONFIGURACIÓN (cambia BASE solo si tu carpeta tiene otro nombre) ----------------
BASE = Path("/content/drive/MyDrive/AgroCebada") if IN_COLAB else Path(os.environ.get("AGC_BASE", "AgroCebada"))
RAW, INTERIM, FEATURES, MODELS, OUTPUTS, QA, SRC, DOCS = [
    BASE / d for d in ["raw", "interim", "features", "models", "outputs", "qa", "src", "docs"]]
for d in [INTERIM, FEATURES, MODELS, OUTPUTS, QA, SRC, DOCS, BASE / "notebooks"]:
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)
CRS_GEO = "EPSG:4326"   # shapefile y rásteres de clima (WGS84)
CRS_MX = "EPSG:6372"    # México ITRF2008 / LCC: topografía, áreas y distancias métricas
N_PLOTS, N_TRAIN, N_PRED = 197, 138, 59
EXPECTED_IDS = {f"AGC_{i:03d}" for i in range(1, N_PLOTS + 1)}
CLEAR_MAX_CLOUD = 10    # solo se usa aquí para CONTAR observaciones despejadas; el filtro real se decide en el NB02

assert RAW.exists() and any(RAW.rglob("*")), f"Primero copia los archivos del reto en {RAW}."

print(f"BASE = {BASE} | Colab: {IN_COLAB} | Python {platform.python_version()} | pandas {pd.__version__}")

Mounted at /content/drive
BASE = /content/drive/MyDrive/AgroCebada | Colab: True | Python 3.13.16 | pandas 2.2.3


In [3]:
# 3 · Funciones de auditoría: cada verificación se registra y termina en qa/audit_report.md
CHECKS, SUMMARY = [], {}

def check(section, name, ok, detail="", warn_only=False, info=False):
    status = "INFO" if info else ("PASS" if ok else ("WARN" if warn_only else "FAIL"))
    rec = {"section": section, "check": name, "status": status, "detail": str(detail)}
    for j, r in enumerate(CHECKS):          # volver a correr una celda sobrescribe sus propias verificaciones
        if (r["section"], r["check"]) == (section, name):
            CHECKS[j] = rec
            break
    else:
        CHECKS.append(rec)
    print(f"[{status}] {section} · {name}" + (f" — {detail}" if detail != "" else ""))
    return ok

def norm(s):
    """Versión ASCII en minúsculas del nombre de una columna (maneja 'área_ha', BOM y espacios)."""
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return s.replace("﻿", "").strip().lower()

def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def md_table(df, max_rows=60):
    """Tabla en Markdown sin dependencias adicionales."""
    df = df.head(max_rows).copy()
    cols = [str(c) for c in df.columns]
    out = ["| " + " | ".join(cols) + " |", "|" + "---|" * len(cols)]
    for _, r in df.iterrows():
        out.append("| " + " | ".join("" if pd.isna(v) else (f"{v:.3f}" if isinstance(v, float) else str(v)) for v in r.values) + " |")
    return "\n".join(out)

def read_csv_any(path, **kw):
    """Lector de CSV tolerante a UTF-8 con o sin BOM y a archivos en Latin-1."""
    try:
        return pd.read_csv(path, encoding="utf-8-sig", **kw)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", **kw)

if "display" not in globals():   # fuera de un cuaderno
    display = print

## 2 · Búsqueda de archivos

Cada archivo dentro de `raw/` se lista y se le calcula su huella (hash). Las **copias exactas** (las carpetas del reto traen dos veces los mismos archivos de clima y topografía) se conservan una sola vez. Si dos archivos de *datos* tienen el mismo nombre pero distinto contenido, la auditoría falla, porque no sabríamos cuál es el correcto.

Los archivos derivados previamente por el equipo (`DataSet_Maestro…`, `DataSet_Listo…` o cualquiera con columna `geometry`) se **ignoran a propósito**: todo el flujo se reconstruye desde los datos originales.

In [4]:
# 4 · Inventario, eliminación de duplicados y clasificación
files = sorted(p for p in RAW.rglob("*") if p.is_file() and not p.name.startswith((".", "~$")))
inv = pd.DataFrame({"path": files})
inv["name"] = inv.path.map(lambda p: p.name)
inv["suffix"] = inv.path.map(lambda p: p.suffix.lower())
inv["size_mb"] = inv.path.map(lambda p: p.stat().st_size / 1e6)
inv["rel"] = inv.path.map(lambda p: str(p.relative_to(RAW)))
inv["md5"] = inv.path.map(md5)

n_before = len(inv)
inv = inv.drop_duplicates("md5", keep="first").reset_index(drop=True)
DATA_SUFFIXES = {".tif", ".tiff", ".csv", ".shp", ".dbf", ".shx", ".prj", ".cpg"}
clash = inv[inv.suffix.isin(DATA_SUFFIXES) & inv.name.duplicated(keep=False)]
check("files", "copias exactas eliminadas", True, f"{n_before - len(inv)} duplicados descartados, {len(inv)} archivos únicos", info=True)
check("files", "ningún archivo de datos con el mismo nombre y distinto contenido", clash.empty, clash.rel.tolist()[:6])

def classify(row):
    n, s, low = row["name"], row["suffix"], row["name"].lower()
    if s == ".shp":
        return "plots_shp"
    if s in (".tif", ".tiff"):
        if re.match(r"prec_\d{4}_\d{2}", low): return "prec"
        if re.match(r"tmin_\d{4}_\d{2}", low): return "tmin"
        if re.match(r"tmax_\d{4}_\d{2}", low): return "tmax"
        if "elev" in low: return "elevation"
        if "pend" in low or "slope" in low: return "slope"
        return "tif_other"
    if s == ".csv":
        if "inventario" in low:
            return "inventory"
        cols = [norm(c) for c in read_csv_any(row["path"], nrows=0).columns]
        if "geometry" in cols:
            return "derived_ignored"
        has_date = any("fecha" in c for c in cols)
        if "rendimiento_t_ha" in cols and not has_date:
            return "labels"
        if has_date:
            tokens = set(re.split(r"[_\W]+", low))
            return "sat_pro" if ("pro" in tokens or "msavi_promedio" in cols) else "sat_basic"
        return "csv_other"
    return "other"

inv["kind"] = inv.apply(classify, axis=1)
FILES = {k: sorted(g.path.tolist()) for k, g in inv.groupby("kind")}
inv.drop(columns="path").to_csv(QA / "file_inventory.csv", index=False)

expected = {"plots_shp": 1, "labels": 1, "sat_pro": 1, "sat_basic": 1, "prec": 48, "tmin": 48, "tmax": 48, "elevation": 1, "slope": 1}
for k, n in expected.items():
    got = len(FILES.get(k, []))
    check("files", f"{k}: se esperaban {n}", got == n, f"encontrados {got}: {[p.name for p in FILES.get(k, [])][:3]}{' …' if got > 3 else ''}")
if FILES.get("derived_ignored"):
    check("files", "tablas derivadas por el equipo ignoradas", True, [p.name for p in FILES["derived_ignored"]], info=True)
display(inv.groupby("kind").agg(n=("name", "size"), size_mb=("size_mb", "sum")).round(2))
SUMMARY["files"] = {k: len(v) for k, v in FILES.items()}

[INFO] files · copias exactas eliminadas — 0 duplicados descartados, 162 archivos únicos
[PASS] files · ningún archivo de datos con el mismo nombre y distinto contenido — []
[PASS] files · plots_shp: se esperaban 1 — encontrados 1: ['Parcelas_Reto_AGC_CONJUNTO.shp']
[PASS] files · labels: se esperaban 1 — encontrados 1: ['ID_area_rendimiento_70_30_Reto_AgroCebada.csv']
[PASS] files · sat_pro: se esperaban 1 — encontrados 1: ['Conjunto_datos_PRO_AgroCebada.csv']
[PASS] files · sat_basic: se esperaban 1 — encontrados 1: ['Conjunto_datos_BASICO_AgroCebada2026.csv']
[PASS] files · prec: se esperaban 48 — encontrados 48: ['PREC_2022_01.tif', 'PREC_2022_02.tif', 'PREC_2022_03.tif'] …
[PASS] files · tmin: se esperaban 48 — encontrados 48: ['Tmin_2022_01.tif', 'Tmin_2022_02.tif', 'Tmin_2022_03.tif'] …
[PASS] files · tmax: se esperaban 48 — encontrados 48: ['Tmax_2022_01.tif', 'Tmax_2022_02.tif', 'Tmax_2022_03.tif'] …
[PASS] files · elevation: se esperaban 1 — encontrados 1: ['Elevacion_INEGI_C

## 3 · Rendimientos: ID, área, rendimiento y CONJUNTO

Verifica lo que establece el reto: 197 identificadores `AGC_001…AGC_197`, 138 parcelas de entrenamiento con rendimiento (t/ha, ciclo abril–octubre 2025) y 59 parcelas de predicción sin rendimiento.

In [5]:
# 5 · Auditoría de los rendimientos
lab = read_csv_any(FILES["labels"][0])
lab.columns = [c.replace("﻿", "").strip() for c in lab.columns]
lab = lab.loc[:, [c for c in lab.columns if c and not c.startswith("Unnamed")]]   # columna vacía al final del CSV
lab["ID_POLIGONO"] = lab["ID_POLIGONO"].astype(str).str.strip()
lab["CONJUNTO"] = lab["CONJUNTO"].astype(str).str.strip().str.upper()
lab["AREA_HA"] = pd.to_numeric(lab["AREA_HA"], errors="coerce")
lab["RENDIMIENTO_T_HA"] = pd.to_numeric(lab["RENDIMIENTO_T_HA"], errors="coerce")

check("labels", "197 filas", len(lab) == N_PLOTS, len(lab))
check("labels", "identificadores únicos", lab.ID_POLIGONO.is_unique)
check("labels", "identificadores AGC_001…AGC_197", set(lab.ID_POLIGONO) == EXPECTED_IDS, sorted(set(lab.ID_POLIGONO) ^ EXPECTED_IDS)[:5])
check("labels", "valores de CONJUNTO", set(lab.CONJUNTO) <= {"ENTRENAMIENTO", "PREDICCION"}, lab.CONJUNTO.value_counts().to_dict())
lab["is_train"] = lab.CONJUNTO.eq("ENTRENAMIENTO")
check("labels", "138 de entrenamiento / 59 de predicción", (lab.is_train.sum(), (~lab.is_train).sum()) == (N_TRAIN, N_PRED),
      f"{lab.is_train.sum()} / {(~lab.is_train).sum()}")
check("labels", "rendimiento presente exactamente en ENTRENAMIENTO", (lab.is_train == lab.RENDIMIENTO_T_HA.notna()).all())
y_tr = lab.loc[lab.is_train, "RENDIMIENTO_T_HA"]
check("labels", "rendimiento entre 0.5 y 10 t/ha", y_tr.between(0.5, 10).all(), f"{y_tr.min():.2f}–{y_tr.max():.2f}")
check("labels", "área > 0", (lab.AREA_HA > 0).all(), f"{lab.AREA_HA.min():.2f}–{lab.AREA_HA.max():.2f} ha")

lab = lab.sort_values("ID_POLIGONO").reset_index(drop=True)
lab.to_parquet(INTERIM / "labels.parquet", index=False)
SUMMARY["labels"] = {"n": len(lab), "n_train": int(lab.is_train.sum()), "yield": y_tr.describe().round(3).to_dict()}
display(y_tr.describe().round(3).to_frame("yield_t_ha").T)

[PASS] labels · 197 filas — 197
[PASS] labels · identificadores únicos
[PASS] labels · identificadores AGC_001…AGC_197 — []
[PASS] labels · valores de CONJUNTO — {'ENTRENAMIENTO': 138, 'PREDICCION': 59}
[PASS] labels · 138 de entrenamiento / 59 de predicción — 138 / 59
[PASS] labels · rendimiento presente exactamente en ENTRENAMIENTO
[PASS] labels · rendimiento entre 0.5 y 10 t/ha — 2.00–5.59
[PASS] labels · área > 0 — 0.35–64.30 ha
            count   mean    std  min  25%   50%   75%   max
yield_t_ha  138.0  3.955  0.851  2.0  3.2  4.34  4.55  5.59


## 4 · Polígonos de las parcelas

Lee el shapefile, normaliza los nombres de campos y los textos (`Cultivo` viene como `CEBADA` y como `Cebada`), elimina la coordenada Z, repara geometrías inválidas, recalcula el área en EPSG:6372 y guarda un centroide métrico (`x_m`, `y_m`) que se usa en todas las distancias a partir de aquí.

In [6]:
# 6 · Auditoría de las parcelas (geoespacial)
import geopandas as gpd
import shapely

shp = FILES["plots_shp"][0]
for ext in [".dbf", ".shx", ".prj"]:
    check("plots", f"archivo complementario {ext} presente", shp.with_suffix(ext).exists())

gdf = gpd.read_file(shp)
rename = {}
for c in gdf.columns:
    k = norm(c)
    if k.startswith("id_polig"): rename[c] = "ID_POLIGONO"
    elif k == "area_ha": rename[c] = "AREA_HA_SHP"
    elif k == "municipio": rename[c] = "Municipio"
    elif k == "estado": rename[c] = "Estado"
    elif k == "conjunto": rename[c] = "CONJUNTO_SHP"
    elif k == "cultivo": rename[c] = "Cultivo"
gdf = gdf.rename(columns=rename)
check("plots", "campos requeridos presentes", {"ID_POLIGONO", "Municipio", "Estado", "CONJUNTO_SHP"} <= set(gdf.columns), list(gdf.columns))

if gdf.crs is None:
    gdf = gdf.set_crs(CRS_GEO)
    check("plots", "SRC definido", False, "faltaba → se asigna EPSG:4326 (el .prj indica WGS84)", warn_only=True)
else:
    check("plots", "el SRC es WGS84", gdf.crs.to_epsg() == 4326, gdf.crs.to_string(), warn_only=True)
    gdf = gdf.to_crs(CRS_GEO)

for c in ["ID_POLIGONO", "Municipio", "Estado", "CONJUNTO_SHP", "Cultivo"]:
    if c in gdf.columns:
        gdf[c] = gdf[c].astype(str).str.strip()
gdf["CONJUNTO_SHP"] = gdf["CONJUNTO_SHP"].str.upper()
if "Cultivo" in gdf.columns:
    check("plots", "etiqueta de cultivo (mayúsculas normalizadas)", True, gdf.Cultivo.value_counts().to_dict(), info=True)
    gdf["Cultivo"] = gdf["Cultivo"].str.title()

had_z = bool(gdf.geometry.has_z.any())
gdf[gdf.geometry.name] = gpd.GeoSeries(shapely.force_2d(np.asarray(gdf.geometry.values, dtype=object)), index=gdf.index, crs=CRS_GEO)
check("plots", "coordenada Z eliminada", True, f"tenía Z: {had_z}", info=True)

n_invalid = int((~gdf.is_valid).sum())
if n_invalid:
    gdf[gdf.geometry.name] = gdf.geometry.make_valid()
check("plots", "geometrías válidas", n_invalid == 0, f"{n_invalid} reparadas con make_valid", warn_only=True)
check("plots", "las geometrías son polígonos", gdf.geom_type.isin(["Polygon", "MultiPolygon"]).all(), gdf.geom_type.value_counts().to_dict())
check("plots", "identificadores únicos e iguales a los de rendimientos", gdf.ID_POLIGONO.is_unique and set(gdf.ID_POLIGONO) == set(lab.ID_POLIGONO))

gdf = gdf.merge(lab[["ID_POLIGONO", "CONJUNTO", "AREA_HA"]], on="ID_POLIGONO", how="left")
check("plots", "CONJUNTO del shapefile = CONJUNTO del CSV de rendimientos", (gdf.CONJUNTO == gdf.CONJUNTO_SHP).all(),
      gdf.loc[gdf.CONJUNTO != gdf.CONJUNTO_SHP, "ID_POLIGONO"].tolist()[:5])

g_mx = gdf.to_crs(CRS_MX)
gdf["area_ha_calc"] = g_mx.area.values / 1e4
rel = (gdf.area_ha_calc - gdf.AREA_HA).abs() / gdf.AREA_HA
check("plots", "área recalculada a menos de 5 % de AREA_HA (≥ 95 % de las parcelas)", (rel <= 0.05).mean() >= 0.95,
      f"diferencia mediana {rel.median():.1%}, máxima {rel.max():.1%} ({gdf.loc[rel.idxmax(), 'ID_POLIGONO']})", warn_only=True)

cent_mx = g_mx.geometry.centroid
gdf["x_m"], gdf["y_m"] = cent_mx.x.values, cent_mx.y.values
cent_geo = cent_mx.to_crs(CRS_GEO)
gdf["lon"], gdf["lat"] = cent_geo.x.values, cent_geo.y.values

plots = gdf[["ID_POLIGONO", "Cultivo", "Municipio", "Estado", "CONJUNTO", "AREA_HA", "area_ha_calc",
             "x_m", "y_m", "lon", "lat", "geometry"]].copy()
plots["is_train"] = plots.CONJUNTO.eq("ENTRENAMIENTO")
plots = gpd.GeoDataFrame(plots, geometry="geometry", crs=CRS_GEO).sort_values("ID_POLIGONO").reset_index(drop=True)

ct = pd.crosstab([plots.Estado, plots.Municipio], plots.CONJUNTO, margins=True)
display(ct)
missing_muni = set(plots.loc[~plots.is_train, "Municipio"]) - set(plots.loc[plots.is_train, "Municipio"])
check("plots", "todo municipio de predicción tiene parcelas de entrenamiento", not missing_muni, missing_muni or "")
SUMMARY["plots"] = {"crs_in": str(gdf.crs), "had_z": had_z, "invalid_fixed": n_invalid,
                    "area_rel_diff_median": round(float(rel.median()), 4),
                    "municipalities": plots.Municipio.nunique(), "states": plots.Estado.value_counts().to_dict()}

[PASS] plots · archivo complementario .dbf presente
[PASS] plots · archivo complementario .shx presente
[PASS] plots · archivo complementario .prj presente
[PASS] plots · campos requeridos presentes — ['Cultivo', 'ID_POLIGONO', 'AREA_HA_SHP', 'Municipio', 'Estado', 'CONJUNTO_SHP', 'geometry']
[PASS] plots · el SRC es WGS84 — EPSG:4326
[INFO] plots · etiqueta de cultivo (mayúsculas normalizadas) — {'CEBADA': 147, 'Cebada': 50}
[INFO] plots · coordenada Z eliminada — tenía Z: True
[PASS] plots · geometrías válidas — 0 reparadas con make_valid
[PASS] plots · las geometrías son polígonos — {'Polygon': 197}
[PASS] plots · identificadores únicos e iguales a los de rendimientos
[PASS] plots · CONJUNTO del shapefile = CONJUNTO del CSV de rendimientos — []
[PASS] plots · área recalculada a menos de 5 % de AREA_HA (≥ 95 % de las parcelas) — diferencia mediana 0.7%, máxima 1.9% (AGC_069)
CONJUNTO                                ENTRENAMIENTO  PREDICCION  All
Estado   Municipio                     

## 5 · Tablas satelitales (PRO = Planet 2025; Básico = Sentinel-2 + Landsat 2022–2025)

**El formato de fecha se detecta a partir de los datos, nunca se supone.** Un valor como `30/01/2025` prueba que el día va primero; uno como `01/30/2025` probaría que va primero el mes. Interpretar con un formato explícito hace que un error falle de forma visible en lugar de intercambiar en silencio día y mes (el archivo PRO hace fallar al lector por omisión de pandas en el 59 % de las filas).

Aquí no se filtra nada. La auditoría reporta la distribución de nubosidad, los valores faltantes (y si coinciden con filas nubladas), los duplicados, la consistencia interna de los estadísticos por parcela (`min ≤ promedio ≤ max`, `std ≥ 0`), qué índices trae realmente cada sensor y cuántas observaciones despejadas tiene cada parcela por mes.

In [7]:
# 7 · Lector satelital genérico y auditoría
def parse_dates(raw):
    s = raw.astype(str).str.strip().str.split(" ").str[0]
    if s.str.fullmatch(r"\d{4}-\d{2}-\d{2}").all():
        return pd.to_datetime(s, format="%Y-%m-%d"), "YYYY-MM-DD", 0, 0
    parts = s.str.extract(r"^(\d{1,2})[/-](\d{1,2})[/-](\d{4})$")
    if parts.isna().any().any():
        bad = s[parts.isna().any(axis=1)].unique()[:5]
        raise ValueError(f"Valores de fecha no reconocidos: {bad}")
    a, b = parts[0].astype(int), parts[1].astype(int)
    a12, b12 = int((a > 12).sum()), int((b > 12).sum())
    if a12 and b12:
        raise ValueError("Formatos de fecha mezclados: hay evidencia de día primero y de mes primero.")
    sep = "-" if s.str.contains("-").any() else "/"
    if b12:
        fmt, label = f"%m{sep}%d{sep}%Y", "MM/DD/YYYY"
    else:
        fmt, label = f"%d{sep}%m{sep}%Y", "DD/MM/YYYY" + ("" if a12 else " (AMBIGUOUS: ningún campo > 12)")
    return pd.to_datetime(s, format=fmt), label, a12, b12

def norm_sensor(x):
    x = str(x).lower()
    if "sentinel" in x or x.startswith("s2"): return "Sentinel-2"
    if "landsat" in x or x.startswith("l8") or x.startswith("l9"): return "Landsat"
    if "planet" in x or "ps" == x: return "Planet"
    return str(x)

def load_and_audit_sat(path, tag, years):
    sec = f"sat_{tag}"
    df = read_csv_any(path, low_memory=False)
    df.columns = [c.replace("﻿", "").strip() for c in df.columns]
    df = df.loc[:, [c for c in df.columns if c and not c.startswith("Unnamed")]]
    idc = next(c for c in df.columns if norm(c).startswith("id_polig"))
    datec = next(c for c in df.columns if "fecha" in norm(c))
    cloudc = next((c for c in df.columns if "nubos" in norm(c) or "cloud" in norm(c)), None)
    sensc = next((c for c in df.columns if norm(c) == "sensor"), None)
    df = df.rename(columns={idc: "ID_POLIGONO"})
    df["ID_POLIGONO"] = df["ID_POLIGONO"].astype(str).str.strip()

    df["date"], fmt, a12, b12 = parse_dates(df[datec])
    check(sec, "formato de fecha detectado a partir de los datos", "AMBIGUOUS" not in fmt,
          f"{fmt}; filas con 1er campo > 12: {a12}, 2o campo > 12: {b12}", warn_only=True)
    check(sec, f"fechas dentro de {years[0]}–{years[-1]}", df.date.dt.year.isin(years).all(),
          f"{df.date.min().date()} → {df.date.max().date()}")
    df["sensor"] = df[sensc].map(norm_sensor) if sensc else tag
    df["cloud"] = pd.to_numeric(df[cloudc], errors="coerce") if cloudc else np.nan
    check(sec, "columnas encontradas (id / fecha / nubosidad / sensor)", cloudc is not None and sensc is not None,
          f"id={idc}, date={datec}, cloud={cloudc}, sensor={sensc}", warn_only=True)

    ids = set(df.ID_POLIGONO)
    check(sec, "cubre las 197 parcelas, sin identificadores desconocidos", ids == EXPECTED_IDS,
          f"{len(ids & EXPECTED_IDS)} parcelas conocidas; desconocidas: {sorted(ids - EXPECTED_IDS)[:3]}; faltantes: {sorted(EXPECTED_IDS - ids)[:3]}")
    dup = int(df.duplicated(["ID_POLIGONO", "date", "sensor"]).sum())
    check(sec, "sin duplicados (parcela, fecha, sensor)", dup == 0, f"{dup} filas duplicadas", warn_only=True)

    stat_cols = [c for c in df.columns if re.search(r"_(promedio|std|max|min)$", c, re.I)]
    indices = sorted({re.sub(r"_(promedio|std|max|min)$", "", c, flags=re.I) for c in stat_cols})
    known = set(stat_cols) | {"ID_POLIGONO", datec, "date", "sensor", "cloud", cloudc, sensc}
    other_num = [c for c in df.columns if c not in known and pd.api.types.is_numeric_dtype(df[c])]
    check(sec, "índices encontrados", True, f"{len(indices)}: {indices}" + (f"; otras columnas numéricas: {other_num}" if other_num else ""), info=True)
    for c in stat_cols + other_num:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    # distribución de nubosidad y patrón de valores faltantes
    bins = pd.cut(df.cloud, [-0.1, 0, 10, 50, 99.999, 100], labels=["0", "(0,10]", "(10,50]", "(50,100)", "100"])
    mean_cols = [f"{i}_promedio" for i in indices if f"{i}_promedio" in df.columns]
    all_nan = df[mean_cols].isna().all(axis=1) if mean_cols else pd.Series(False, index=df.index)
    cloud_tab = pd.DataFrame({"rows": bins.value_counts().sort_index(),
                              "share_rows_all_indices_NaN": all_nan.groupby(bins, observed=False).mean().round(3)})
    display(cloud_tab)
    check(sec, "filas con todos los índices vacíos", True, f"{all_nan.mean():.1%} de las filas", info=True)

    # qué índices trae cada sensor (proporción no nula de *_promedio)
    cov = df.groupby("sensor")[mean_cols].apply(lambda g: g.notna().mean()).T.round(2) if mean_cols else pd.DataFrame()
    display(cov)
    check(sec, "sensores", True, df.sensor.value_counts().to_dict(), info=True)

    # consistencia interna de los estadísticos por parcela
    viol = {}
    for i in indices:
        cols = {s: f"{i}_{s}" for s in ["promedio", "std", "max", "min"]}
        if all(c in df.columns for c in cols.values()):
            v = df[list(cols.values())].dropna()
            bad = (v[cols["min"]] > v[cols["promedio"]] + 1e-6) | (v[cols["promedio"]] > v[cols["max"]] + 1e-6) | (v[cols["std"]] < 0)
            if bad.any(): viol[i] = int(bad.sum())
    check(sec, "min ≤ promedio ≤ max and std ≥ 0", not viol, viol or "", warn_only=True)

    # observaciones despejadas por parcela y mes (nubosidad ≤ CLEAR_MAX_CLOUD y con índice)
    clear = df[(df.cloud.fillna(0) <= CLEAR_MAX_CLOUD) & ~all_nan]
    per = clear.groupby(["sensor", "ID_POLIGONO", clear.date.dt.to_period("M")]).size()
    per = per.unstack("date", fill_value=0)
    stats = per.groupby(level="sensor").agg(["median", "min"]).T if len(per) else pd.DataFrame()
    stats.to_csv(QA / f"sat_clear_obs_per_month_{tag}.csv")
    month_median = per.groupby(level="sensor").median().T if len(per) else pd.DataFrame()
    display(month_median.tail(14))

    keep = ["ID_POLIGONO", "date", "sensor", "cloud"] + stat_cols + other_num
    out = df[keep].sort_values(["ID_POLIGONO", "sensor", "date"]).reset_index(drop=True)
    SUMMARY[sec] = {"file": Path(path).name, "rows": len(df), "date_format": fmt,
                    "date_range": [str(df.date.min().date()), str(df.date.max().date())],
                    "sensors": df.sensor.value_counts().to_dict(), "indices": indices, "other_numeric": other_num,
                    "cloud_bins": cloud_tab["rows"].astype(int).to_dict(), "all_nan_share": round(float(all_nan.mean()), 4),
                    "duplicates": dup, "consistency_violations": viol,
                    "index_coverage_by_sensor": cov.to_dict() if len(cov) else {}}
    return out

In [8]:
# 8 · Conjunto PRO (Planet, 2025)
pro = load_and_audit_sat(FILES["sat_pro"][0], "pro", years=[2025])
pro.to_parquet(INTERIM / "pro_obs.parquet", index=False)
print(pro.shape)

[PASS] sat_pro · formato de fecha detectado a partir de los datos — DD/MM/YYYY; filas con 1er campo > 12: 28304, 2o campo > 12: 0
[PASS] sat_pro · fechas dentro de 2025–2025 — 2025-01-01 → 2025-12-30
[PASS] sat_pro · columnas encontradas (id / fecha / nubosidad / sensor) — id=ID_POLIGONO, date=fecha_captura, cloud=porcentaje_nubosidad, sensor=sensor
[PASS] sat_pro · cubre las 197 parcelas, sin identificadores desconocidos — 197 parcelas conocidas; desconocidas: []; faltantes: []
[PASS] sat_pro · sin duplicados (parcela, fecha, sensor) — 0 filas duplicadas
[INFO] sat_pro · índices encontrados — 4: ['evi', 'lai', 'msavi', 'ndvi']
           rows  share_rows_all_indices_NaN
cloud                                      
0         30317                         0.0
(0,10]     1391                         0.0
(10,50]    1730                         0.0
(50,100)   2309                         0.0
100       12057                         1.0
[INFO] sat_pro · filas con todos los índices vacíos — 25

In [9]:
# 9 · Conjunto Básico (Sentinel-2 + Landsat, 2022–2025)
if FILES.get("sat_basic"):
    basic = load_and_audit_sat(FILES["sat_basic"][0], "basic", years=[2022, 2023, 2024, 2025])
    basic.to_parquet(INTERIM / "basic_obs.parquet", index=False)
    print(basic.shape)
else:
    check("sat_basic", "conjunto Básico encontrado", False, "No se detectó el CSV Básico en raw/: cópialo y vuelve a correr las celdas 4 y 9.")

[PASS] sat_basic · formato de fecha detectado a partir de los datos — DD/MM/YYYY; filas con 1er campo > 12: 65841, 2o campo > 12: 0
[PASS] sat_basic · fechas dentro de 2022–2025 — 2022-01-01 → 2025-12-31
[PASS] sat_basic · columnas encontradas (id / fecha / nubosidad / sensor) — id=ID_POLIGONO, date=fecha_captura, cloud=porcentaje_nubosidad, sensor=sensor
[PASS] sat_basic · cubre las 197 parcelas, sin identificadores desconocidos — 197 parcelas conocidas; desconocidas: []; faltantes: []
[PASS] sat_basic · sin duplicados (parcela, fecha, sensor) — 0 filas duplicadas
[INFO] sat_basic · índices encontrados — 23: ['crc', 'dswi2', 'dswi3', 'dswi4', 'dswi5', 'emsi', 'enddi', 'endwi', 'evi', 'fapar', 'lai', 'mcrc', 'mra', 'msi', 'nddi', 'ndsvi', 'ndti', 'ndvi', 'ndwi', 'savi', 'srndi', 'sti', 'vi6t']
           rows  share_rows_all_indices_NaN
cloud                                      
0         63823                         0.0
(0,10]     3276                         0.0
(10,50]    3858    

## 6 · Rásteres de clima y topografía

Inventario de cada GeoTIFF (sistema de referencia, malla, valor sin dato, rango de valores), integridad de las 48 capas mensuales por variable y una **prueba de plausibilidad en los centroides de las parcelas**: la precipitación de abril a octubre de 2025 debe ser de unos cientos de mm. Además, cada parcela recibe el identificador de la celda climática de 0.05° donde cae (`clim_cell`). La extracción de variables se hace en el cuaderno 04.

In [10]:
# 10 · Auditoría de rásteres (geoespacial)
import rasterio
from rasterio.warp import transform as warp_transform

rows = []
for kind in ["prec", "tmin", "tmax", "elevation", "slope"]:
    for p in FILES.get(kind, []):
        with rasterio.open(p) as src:
            a = src.read(1, masked=True)
            m = re.search(r"(\d{4})_(\d{2})", p.name)
            rows.append({"kind": kind, "file": p.name,
                         "year": int(m.group(1)) if m else None, "month": int(m.group(2)) if m else None,
                         "crs": src.crs.to_string() if src.crs else None, "epsg": src.crs.to_epsg() if src.crs else None,
                         "width": src.width, "height": src.height, "res_x": src.res[0], "res_y": src.res[1],
                         "x0": src.transform.c, "y0": src.transform.f,
                         "nodata": src.nodata, "dtype": src.dtypes[0], "valid_px": int(a.count()),
                         "vmin": float(a.min()), "vmax": float(a.max()), "vmean": float(a.mean())})
rinv = pd.DataFrame(rows)
rinv.to_csv(QA / "raster_inventory.csv", index=False)

months = {(y, m) for y in range(2022, 2026) for m in range(1, 13)}
for kind in ["prec", "tmin", "tmax"]:
    r = rinv[rinv["kind"] == kind]
    got = set(zip(r.year, r.month))
    check("rasters", f"{kind}: 48 meses 2022-01…2025-12", got == months, f"faltan: {sorted(months - got)[:4]}")
# Alineación de mallas. El tamaño de píxel se guarda como float32 (0.05000000074505806), así que exigir
# igualdad exacta es demasiado estricto: se comparan los orígenes en unidades de píxel, con tolerancia de 1 % de píxel.
clim = rinv[rinv["kind"].isin(["prec", "tmin", "tmax"])].copy()
ref = clim[clim["kind"] == "prec"].iloc[0]
clim["offset_px"] = np.maximum((clim.x0 - ref.x0).abs(), (clim.y0 - ref.y0).abs()) / ref.res_x
clim["res_rel_diff"] = (clim.res_x - ref.res_x).abs() / ref.res_x
grid_by_kind = clim.groupby("kind").agg(crs=("crs", "first"), width=("width", "max"), height=("height", "max"),
                                        x0=("x0", "first"), y0=("y0", "first"), res=("res_x", "first"),
                                        n_distinct_origins=("x0", lambda s: s.round(9).nunique()),
                                        max_offset_px=("offset_px", "max"), max_res_rel_diff=("res_rel_diff", "max"))
display(grid_by_kind)
same_shape = clim[["crs", "width", "height"]].drop_duplicates().shape[0] == 1
GRID_ALIGNED = bool(same_shape and clim.offset_px.max() < 0.01 and clim.res_rel_diff.max() < 1e-4)
check("rasters", "capas de clima alineadas en una sola malla (tolerancia de 1 % de píxel)", GRID_ALIGNED,
      "; ".join(f"{k}: origen ({r.x0:.6f}, {r.y0:.6f}), desfase máximo {r.max_offset_px:.4f} px"
                for k, r in grid_by_kind.iterrows())
      + ("" if GRID_ALIGNED else " → mallas distintas: el NB04 extrae cada variable en su propia malla; clim_cell usa la malla de precipitación"),
      warn_only=True)
check("rasters", "ningún valor sin dato mezclado con los valores (mín. > -1000)", (rinv.vmin > -1000).all(),
      rinv.loc[rinv.vmin <= -1000, "file"].tolist()[:4])
check("rasters", "precipitación ≥ 0", (rinv.loc[rinv["kind"] == "prec", "vmin"] >= 0).all())
tm = rinv[rinv["kind"].isin(["tmin", "tmax"])].pivot_table(index=["year", "month"], columns="kind", values="vmean")
check("rasters", "Tmáx media regional > Tmín todos los meses", (tm["tmax"] > tm["tmin"]).all())
topo = rinv[rinv["kind"].isin(["elevation", "slope"])]
check("rasters", "el SRC de la topografía es EPSG:6372", (topo.epsg == 6372).all(), topo[["file", "crs"]].to_dict("records"), warn_only=True)

from rasterio.mask import mask as rio_mask
from rasterio.windows import Window
from shapely.geometry import mapping

def sample_at(path, lon, lat):
    """Valor de la banda 1 en cada punto (lon/lat en EPSG:4326); NaN donde no hay dato."""
    with rasterio.open(path) as src:
        xs, ys = (lon, lat) if src.crs.to_epsg() == 4326 else warp_transform(CRS_GEO, src.crs, list(lon), list(lat))
        return np.array([float(np.ma.filled(v.astype("float64"), np.nan)[0]) for v in src.sample(list(zip(xs, ys)), masked=True)])

def raster_path(kind, y, m):
    return next(p for p in FILES[kind] if f"{y}_{m:02d}" in p.name)

_geoms = {}
def geoms_in(crs):
    key = crs.to_string()
    if key not in _geoms:
        _geoms[key] = list(plots.to_crs(crs).geometry.values)
    return _geoms[key]

def fallback_value(path, i, max_px=3):
    """Para una parcela cuyo centroide cae en un píxel sin dato: media de los píxeles válidos que toca su polígono;
    si no toca ninguno, el píxel válido más cercano a menos de max_px píxeles del centroide."""
    with rasterio.open(path) as src:
        g = geoms_in(src.crs)[i]
        try:
            arr, _ = rio_mask(src, [mapping(g)], crop=True, all_touched=True, filled=False)
            if arr.count() > 0:
                return float(arr.mean()), f"polygon_mean ({arr.count()} px)"
        except ValueError:                      # polígono fuera de la extensión del ráster
            pass
        c = g.centroid
        r0, c0 = src.index(c.x, c.y)
        a = src.read(1, window=Window(c0 - max_px, r0 - max_px, 2 * max_px + 1, 2 * max_px + 1),
                     masked=True, boundless=True)
        if a.count() == 0:
            return np.nan, "unresolved"
        rr, cc = np.nonzero(~np.ma.getmaskarray(a))
        k = int(np.argmin((rr - max_px) ** 2 + (cc - max_px) ** 2))
        return float(a[rr[k], cc[k]]), f"nearest_valid ({np.hypot(rr[k] - max_px, cc[k] - max_px):.1f} px)"

LAYERS = {f"prec_2025_{m:02d}": raster_path("prec", 2025, m) for m in range(1, 13)}
LAYERS.update({f"tmax_2025_{m:02d}": raster_path("tmax", 2025, m) for m in range(6, 12)})
LAYERS.update({f"tmin_2025_{m:02d}": raster_path("tmin", 2025, m) for m in range(6, 12)})
LAYERS.update({"elevation": FILES["elevation"][0], "slope": FILES["slope"][0]})

lon, lat = plots.lon.values, plots.lat.values
SAMP = pd.DataFrame({k: sample_at(p, lon, lat) for k, p in LAYERS.items()}, index=plots.ID_POLIGONO.values)

id_pos = {pid: i for i, pid in enumerate(plots.ID_POLIGONO)}
fixes = []
for layer in SAMP.columns:
    for pid in SAMP.index[SAMP[layer].isna()]:
        v, how = fallback_value(LAYERS[layer], id_pos[pid])
        SAMP.loc[pid, layer] = v
        r = plots.iloc[id_pos[pid]]
        fixes.append({"ID_POLIGONO": pid, "Municipio": r.Municipio, "is_train": bool(r.is_train),
                      "lon": round(r.lon, 6), "lat": round(r.lat, 6), "area_ha": r.AREA_HA,
                      "layer": layer, "value_after_fix": v, "method": how})
nan_fix = pd.DataFrame(fixes, columns=["ID_POLIGONO", "Municipio", "is_train", "lon", "lat", "area_ha",
                                       "layer", "value_after_fix", "method"])
nan_fix.to_csv(QA / "centroid_nodata_fixes.csv", index=False)

if len(nan_fix):
    nan_fix["variable"] = nan_fix.layer.str.split("_").str[0]
    print("Parcelas cuyo centroide cae en un píxel sin dato (por variable):")
    display(nan_fix.groupby(["variable", "ID_POLIGONO", "Municipio"])
                   .agg(n_layers=("layer", "size"), method=("method", "first"),
                        value_after_fix=("value_after_fix", "mean")).round(3))
    detail = "; ".join(f"{v}: {sorted(g.ID_POLIGONO.unique())}" for v, g in nan_fix.groupby("variable"))
else:
    detail = ""
check("rasters", "todos los centroides caen en un píxel válido", nan_fix.empty, detail, warn_only=True)
unresolved = nan_fix[nan_fix.method == "unresolved"]
check("rasters", "todas las parcelas tienen valor en todas las capas (tras usar el polígono o el píxel más cercano)",
      bool(SAMP.notna().all().all()),
      f"SIN RESOLVER: {unresolved[['ID_POLIGONO', 'layer']].values.tolist()[:10]}" if len(unresolved)
      else (f"{nan_fix.ID_POLIGONO.nunique()} parcela(s) corregida(s): {nan_fix.method.str.split(' ').str[0].value_counts().to_dict()}"
            if len(nan_fix) else "no fue necesario"))

prec_aprOct = SAMP[[f"prec_2025_{m:02d}" for m in range(4, 11)]].sum(axis=1, min_count=7).to_numpy()
prec_year = SAMP[[f"prec_2025_{m:02d}" for m in range(1, 13)]].sum(axis=1, min_count=12).to_numpy()
tmax_jjason = SAMP[[f"tmax_2025_{m:02d}" for m in range(6, 12)]].mean(axis=1, skipna=False).to_numpy()
tmin_jjason = SAMP[[f"tmin_2025_{m:02d}" for m in range(6, 12)]].mean(axis=1, skipna=False).to_numpy()
elev, slope = SAMP["elevation"].to_numpy(), SAMP["slope"].to_numpy()

check("rasters", "precipitación abr–oct 2025 plausible en las parcelas (150–1500 mm)",
      np.nanmin(prec_aprOct) >= 150 and np.nanmax(prec_aprOct) <= 1500,
      f"{np.nanmin(prec_aprOct):.0f}–{np.nanmax(prec_aprOct):.0f} mm (total 2025 {np.nanmin(prec_year):.0f}–{np.nanmax(prec_year):.0f} mm)", warn_only=True)
check("rasters", "Tmáx / Tmín media jun–nov 2025 en las parcelas", True,
      f"Tmax {np.nanmin(tmax_jjason):.1f}–{np.nanmax(tmax_jjason):.1f} °C; Tmin {np.nanmin(tmin_jjason):.1f}–{np.nanmax(tmin_jjason):.1f} °C", info=True)
check("rasters", "elevación de las parcelas plausible para el Altiplano (1800–3300 m)",
      np.nanmin(elev) >= 1800 and np.nanmax(elev) <= 3300, f"{np.nanmin(elev):.0f}–{np.nanmax(elev):.0f} m", warn_only=True)
check("rasters", "pendiente de las parcelas entre 0 y 45°", np.nanmin(slope) >= 0 and np.nanmax(slope) <= 45,
      f"{np.nanmin(slope):.1f}–{np.nanmax(slope):.1f}°", warn_only=True)

# celda climática de cada parcela (malla de 0.05°) → grupos para validación espacial
with rasterio.open(raster_path("prec", 2025, 1)) as src:
    rc = [src.index(x, y) for x, y in zip(lon, lat)]
    plots["clim_cell"] = [r * src.width + c for r, c in rc]
n_cells, n_cells_tr = plots.clim_cell.nunique(), plots.loc[plots.is_train, "clim_cell"].nunique()
check("rasters", "celdas climáticas de 0.05° distintas entre las parcelas", True, f"{n_cells} en total, {n_cells_tr} en entrenamiento", info=True)

SUMMARY["rasters"] = {"layers": rinv["kind"].value_counts().to_dict(),
                      "climate_grid_aligned": GRID_ALIGNED,
                      "climate_grid": grid_by_kind[["x0", "y0", "res", "max_offset_px"]].round(8).to_dict("index"),
                      "centroid_nodata": nan_fix[["ID_POLIGONO", "layer", "method"]].to_dict("records"),
                      "prec_aprOct_2025_mm": [round(float(np.nanmin(prec_aprOct))), round(float(np.nanmax(prec_aprOct)))],
                      "elev_m": [round(float(np.nanmin(elev))), round(float(np.nanmax(elev)))],
                      "n_climate_cells": int(n_cells), "n_climate_cells_train": int(n_cells_tr)}
display(rinv.groupby("kind")[["crs", "width", "height", "res_x", "nodata"]].first())

[PASS] rasters · prec: 48 meses 2022-01…2025-12 — faltan: []
[PASS] rasters · tmin: 48 meses 2022-01…2025-12 — faltan: []
[PASS] rasters · tmax: 48 meses 2022-01…2025-12 — faltan: []
            crs  width  height         x0         y0   res  n_distinct_origins  max_offset_px  max_res_rel_diff
kind                                                                                                           
prec  EPSG:4326     65      72 -99.949999  21.400000  0.05                   1       0.000000               0.0
tmax  EPSG:4326     65      72 -99.949999  21.399999  0.05                   1       0.000006               0.0
tmin  EPSG:4326     65      72 -99.949999  21.399999  0.05                   1       0.000006               0.0
[PASS] rasters · capas de clima alineadas en una sola malla (tolerancia de 1 % de píxel) — prec: origen (-99.949999, 21.400000), desfase máximo 0.0000 px; tmax: origen (-99.949999, 21.399999), desfase máximo 0.0000 px; tmin: origen (-99.949999, 21.399999), 

In [11]:
# 11 · Guardar la tabla de parcelas (ya con clim_cell)
plots.to_parquet(INTERIM / "plots.parquet", index=False)
plots.drop(columns="geometry").to_csv(INTERIM / "plots_attributes.csv", index=False)
print(plots.drop(columns="geometry").head())

  ID_POLIGONO Cultivo     Municipio    Estado       CONJUNTO  AREA_HA  area_ha_calc           x_m            y_m        lon        lat  is_train  clim_cell
0     AGC_001  Cebada   Calpulalpan  Tlaxcala  ENTRENAMIENTO     7.61      7.568578  2.859661e+06  842453.681778 -98.562080  19.542101      True       2432
1     AGC_002  Cebada  Chignahuapan    Puebla  ENTRENAMIENTO     2.31      2.293437  2.891080e+06  878445.661544 -98.252898  19.861052      True       1983
2     AGC_003  Cebada   Singuilucan   Hidalgo     PREDICCION    18.68     18.549386  2.878004e+06  891387.367084 -98.374996  19.981372     False       1851
3     AGC_004  Cebada   Calpulalpan  Tlaxcala  ENTRENAMIENTO     8.00      7.951018  2.862220e+06  846473.738521 -98.536706  19.577960      True       2368
4     AGC_005  Cebada  Chignahuapan    Puebla  ENTRENAMIENTO     2.27      2.257240  2.892005e+06  878239.093966 -98.244096  19.858961      True       1984


## 7 · Estructura del rendimiento y referencia de ubicación

¿Cuánto del rendimiento se puede predecir sabiendo solo **dónde** está la parcela, sin datos satelitales ni climáticos? Validación dejando una fuera (LOO) sobre las 138 parcelas de entrenamiento:

- media general · media municipal · rendimiento medio de las *k* parcelas de entrenamiento más cercanas (distancia métrica, EPSG:6372).

Esta cifra es el umbral que todo modelo debe superar. Se vuelve a medir en los pliegues fijos de validación cruzada del cuaderno 05 (LOO con el mejor *k* elegido aquí es ligeramente optimista).

In [12]:
# 12 · Auditoría del rendimiento y referencias de ubicación
d = pd.DataFrame(plots.drop(columns="geometry")).merge(lab[["ID_POLIGONO", "RENDIMIENTO_T_HA"]], on="ID_POLIGONO")
t = d[d.is_train].reset_index(drop=True)
y = t.RENDIMIENTO_T_HA.to_numpy()

on_half_grid = np.isclose(y * 2, np.round(y * 2))
vc = pd.Series(y).value_counts()
check("yield", "valores en múltiplos de 0.5 t/ha (probablemente declarados o redondeados)", True,
      f"{on_half_grid.sum()}/{len(y)} ({on_half_grid.mean():.0%}); más frecuentes: {vc.head(4).to_dict()}", info=True)
by_muni = (d.groupby("Municipio")
             .agg(n_train=("is_train", "sum"), n_pred=("is_train", lambda s: int((~s).sum())),
                  yield_mean=("RENDIMIENTO_T_HA", "mean"), yield_std=("RENDIMIENTO_T_HA", "std"),
                  yield_min=("RENDIMIENTO_T_HA", "min"), yield_max=("RENDIMIENTO_T_HA", "max"))
             .sort_values("n_train", ascending=False).round(3))
display(by_muni)

def metrics(y_true, y_pred):
    e = y_true - y_pred
    return {"rmse": float(np.sqrt(np.mean(e ** 2))), "mae": float(np.mean(np.abs(e))),
            "r2": float(1 - np.sum(e ** 2) / np.sum((y_true - y_true.mean()) ** 2))}

res = {}
n = len(y)
res["global_mean_LOO"] = metrics(y, (y.sum() - y) / (n - 1))
g = t.groupby("Municipio").RENDIMIENTO_T_HA
cnt, tot = g.transform("count").to_numpy(), g.transform("sum").to_numpy()
glob_loo = (y.sum() - y) / (n - 1)
muni_loo = np.where(cnt > 1, (tot - y) / np.maximum(cnt - 1, 1), glob_loo)
res["municipality_mean_LOO"] = metrics(y, muni_loo)

XY = t[["x_m", "y_m"]].to_numpy()
D = np.sqrt(((XY[:, None, :] - XY[None, :, :]) ** 2).sum(-1))
np.fill_diagonal(D, np.inf)
order = np.argsort(D, axis=1)
K_MAX = 30
knn = {k: metrics(y, y[order[:, :k]].mean(1)) for k in range(1, K_MAX + 1)}
best_k = min(knn, key=lambda k: knn[k]["rmse"])
res["knn_LOO_by_k"] = {int(k): {m: round(v, 4) for m, v in r.items()} for k, r in knn.items()}
res["knn_best"] = {"k": int(best_k), **knn[best_k]}

P = d[~d.is_train][["x_m", "y_m"]].to_numpy()
dist_pred = np.sqrt(((P[:, None, :] - XY[None, :, :]) ** 2).sum(-1)).min(1) / 1000
res["pred_to_nearest_train_km"] = {"median": float(np.median(dist_pred)), "max": float(dist_pred.max())}
res["note"] = "LOO sobre 138 parcelas de entrenamiento; el mejor k se elige con el mismo LOO (optimista). Referencia oficial: pliegues fijos del NB05."
(MODELS / "baselines_location.json").write_text(json.dumps(res, indent=2))

tab = pd.DataFrame({k: v for k, v in res.items() if k in ["global_mean_LOO", "municipality_mean_LOO", "knn_best"]}).T
display(tab.round(3))
check("yield", "referencia a superar (k vecinos más cercanos)", True,
      f"k={best_k}: RMSE {knn[best_k]['rmse']:.3f} t/ha, MAE {knn[best_k]['mae']:.3f}, R² {knn[best_k]['r2']:.3f}"
      + (" (k en el límite de búsqueda)" if best_k == K_MAX else ""), info=True)
check("yield", "las parcelas de predicción están cerca de las de entrenamiento", True,
      f"parcela de entrenamiento más cercana: mediana {np.median(dist_pred):.2f} km, máximo {dist_pred.max():.2f} km", info=True)
SUMMARY["baselines"] = {"global_rmse": round(res["global_mean_LOO"]["rmse"], 3),
                        "municipality": {k: round(v, 3) for k, v in res["municipality_mean_LOO"].items()},
                        "knn_best": {k: (round(v, 3) if isinstance(v, float) else v) for k, v in res["knn_best"].items()}}

[INFO] yield · valores en múltiplos de 0.5 t/ha (probablemente declarados o redondeados) — 45/138 (33%); más frecuentes: {3.0: 19, 3.5: 13, 2.5: 11, 4.52: 5}
                               n_train  n_pred  yield_mean  yield_std  yield_min  yield_max
Municipio                                                                                  
Chignahuapan                        71      29       4.477      0.584       2.15       5.59
Calpulalpan                         30      14       3.000      0.371       2.50       3.50
Singuilucan                         16       7       3.081      0.434       2.00       3.70
Almoloya                            10       4       4.521      0.188       4.14       4.87
Cuautepec de Hinojosa                5       4       4.598      0.151       4.40       4.79
Emiliano Zapata                      3       0       3.500      0.000       3.50       3.50
Apan                                 1       0       3.000        NaN       3.00       3.00
Nanacamilpa de

## 8 · Reporte de auditoría y registro del entorno

In [13]:
# 13 · Escribir qa/audit_report.md y el registro de versiones del entorno
chk = pd.DataFrame(CHECKS)
counts = chk.status.value_counts().to_dict()
lines = [
    "# Reporte de auditoría: cuaderno 01",
    f"Generado: {datetime.now():%Y-%m-%d %H:%M} · Colab: {IN_COLAB} · Python {platform.python_version()} · pandas {pd.__version__}",
    "",
    f"**Conteo de estados:** {counts}",
    "",
    "## Verificaciones",
    md_table(chk, max_rows=500),
    "",
    "## Resumen por conjunto de datos",
]
for k, v in SUMMARY.items():
    lines += [f"### {k}", "```json", json.dumps(v, indent=2, ensure_ascii=False, default=str), "```", ""]
lines += ["## Rendimiento por municipio (parcelas de entrenamiento)", md_table(by_muni.reset_index()), ""]
(QA / "audit_report.md").write_text("\n".join(lines), encoding="utf-8")

req = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(QA / "entorno_pip_freeze.txt").write_text(req)

print(f"Conteo de estados: {counts}")
problems = chk[chk.status.isin(["FAIL", "WARN"])]
if len(problems):
    print("\nRevisa estos puntos antes de continuar:")
    display(problems)
else:
    print("Todas las verificaciones pasaron.")
print(f"\nReporte: {QA / 'audit_report.md'}")

Conteo de estados: {'PASS': 53, 'INFO': 14, 'WARN': 1}

Revisa estos puntos antes de continuar:
    section                                         check status                                    detail
58  rasters  todos los centroides caen en un píxel válido   WARN  slope: ['AGC_023', 'AGC_164', 'AGC_170']

Reporte: /content/drive/MyDrive/AgroCebada/qa/audit_report.md


## Resultado y siguiente paso

1. Abre `qa/audit_report.md`. Todo **FAIL** debe corregirse (casi siempre es un archivo faltante o duplicado en `raw/`) y volver a correr el cuaderno. Los **WARN** están documentados en el reporte y no bloquean.
2. Sigue el **cuaderno 02**: limpieza y suavizado de las series satelitales.